# AnnoMI-simple EDA

Goal: understand the dataset (133 expert-annotated motivational interviewing transcripts) well enough to design an MI review app: audio -> transcript (Content Understanding) -> adherence analysis.

Sections: 1) structure and quality, 2) conversation-level stats, 3) annotation labels, 4) therapist -> client dynamics, 5) per-conversation MI metrics (high vs low), 6) sizing for LLM/CU design.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

# keep_default_na=False: "n/a" is a real label value here, not a missing value
df = pd.read_csv(Path("AnnoMI-simple.csv"), keep_default_na=False)
df.shape

(9699, 11)

## 1. Structure and data quality

In [2]:
pd.to_timedelta(df["timestamp"], errors="coerce")

0      0 days 00:00:13
1      0 days 00:00:24
2      0 days 00:00:25
3      0 days 00:00:34
4      0 days 00:00:34
             ...      
9694   0 days 00:26:45
9695   0 days 00:26:49
9696   0 days 00:26:52
9697   0 days 00:26:54
9698   0 days 00:26:58
Name: timestamp, Length: 9699, dtype: timedelta64[us]

In [4]:
df = df.sort_values(["transcript_id", "utterance_id"]).reset_index(drop=True)
df["ts"] = pd.to_timedelta(df["timestamp"], errors="coerce")
df["words"] = df["utterance_text"].fillna("").str.split().str.len()

print("rows:", len(df), "| transcripts:", df["transcript_id"].nunique())
print("duplicate rows:", df.duplicated().sum())
print("\nnulls per column:")
print(df.isna().sum()[lambda s: s > 0])
print("\nunparseable timestamps:", df["ts"].isna().sum())

g = df.groupby("transcript_id")
print("\ntranscripts with >1 mi_quality:", (g["mi_quality"].nunique() > 1).sum())
print("transcripts with >1 video_url:", (g["video_url"].nunique() > 1).sum())
print("transcripts with non-contiguous utterance_id:",
      (g["utterance_id"].apply(lambda s: (s.diff().dropna() != 1).any())).sum())
print("transcripts with non-monotonic timestamps:",
      (g["ts"].apply(lambda s: (s.diff().dropna() < pd.Timedelta(0)).any())).sum())
df.head(3)

rows: 9699 | transcripts: 133
duplicate rows: 0

nulls per column:
Series([], dtype: int64)

unparseable timestamps: 0

transcripts with >1 mi_quality: 0
transcripts with >1 video_url: 0
transcripts with non-contiguous utterance_id: 0
transcripts with non-monotonic timestamps: 8


,transcript_id,mi_quality,video_title,video_url,topic,utterance_id,interlocutor,timestamp,utterance_text,main_therapist_behaviour,client_talk_type,ts,words
0,0,high,"NEW VIDEO: Brief intervention: ""Barbara""",https://www.youtube.com/watch?v=PaSKcfTmFEk,reducing alcohol consumption,0,therapist,00:00:13,Thanks for filling it out. We give this form to everyone once a year regardless of why they come in. It helps us pro...,question,n/a,0 days 00:00:13,39
1,0,high,"NEW VIDEO: Brief intervention: ""Barbara""",https://www.youtube.com/watch?v=PaSKcfTmFEk,reducing alcohol consumption,1,client,00:00:24,Sure.,n/a,neutral,0 days 00:00:24,1
2,0,high,"NEW VIDEO: Brief intervention: ""Barbara""",https://www.youtube.com/watch?v=PaSKcfTmFEk,reducing alcohol consumption,2,therapist,00:00:25,"So, let's see. It looks that you put-- You drink alcohol at least four times a week on average-",therapist_input,n/a,0 days 00:00:25,19


## 2. Conversation-level stats

In [11]:
conv = df.groupby("transcript_id").agg(
    mi_quality=("mi_quality", "first"),
    title=("video_title", "first"),
    topic=("topic", lambda s: s.iloc[0].strip()),
    n_utts=("utterance_id", "count"),
    words=("words", "sum"),
    duration_min=("ts", lambda s: s.max().total_seconds() / 60),
)

print(conv["mi_quality"].value_counts().to_string())
print()
print(conv.groupby("mi_quality")[["n_utts", "words", "duration_min"]].describe().T.round(1).to_string())

mi_quality
high    110
low      23

mi_quality            high     low
n_utts       count   110.0    23.0
             mean     80.4    37.4
             std      89.4    38.8
             min       7.0     6.0
             25%      31.5    19.0
             50%      52.0    23.0
             75%      90.5    34.5
             max     598.0   173.0
words        count   110.0    23.0
             mean   1242.2   730.0
             std    1133.6   493.5
             min     102.0   119.0
             25%     529.0   412.0
             50%     951.5   589.0
             75%    1533.5   891.0
             max    7347.0  2014.0
duration_min count   110.0    23.0
             mean      8.4     4.3
             std       7.6     2.9
             min       1.0     0.8
             25%       3.7     2.2
             50%       6.0     3.4
             75%       9.3     5.3
             max      37.8    11.3


In [12]:
print("distinct source videos:", df["video_url"].nunique())
print("\ntranscripts per topic (top 20):")
print(conv["topic"].value_counts().head(20).to_string())
print("\nmi_quality by topic (top 10 topics):")
top = conv["topic"].value_counts().head(10).index
print(pd.crosstab(conv.loc[conv["topic"].isin(top), "topic"], conv["mi_quality"]).to_string())

distinct source videos: 119

transcripts per topic (top 20):
topic
reducing alcohol consumption                                   23
smoking cessation                                              19
taking medicine / following medical procedure                   8
reducing recidivism                                             7
reducing drug use                                               7
more exercise / increasing activity                             7
weight loss                                                     6
asthma management                                               5
unidentifiable                                                  5
diabetes management                                             4
compliance with rules                                           4
better oral health                                              2
reducing gambling                                               2
more exercise / increasing activity; weight loss                2
providing

## 3. Annotation labels

In [7]:
ther = df[df["interlocutor"] == "therapist"]
cli = df[df["interlocutor"] == "client"]

print("interlocutor share:", df["interlocutor"].value_counts(normalize=True).round(3).to_dict())

print("\nTherapist behaviour (row %, overall and by mi_quality):")
tb = pd.concat(
    [ther["main_therapist_behaviour"].value_counts(normalize=True).rename("all"),
     pd.crosstab(ther["main_therapist_behaviour"], ther["mi_quality"], normalize="columns")],
    axis=1,
).round(3)
print(tb.to_string())

print("\nClient talk type (overall and by mi_quality):")
ct = pd.concat(
    [cli["client_talk_type"].value_counts(normalize=True).rename("all"),
     pd.crosstab(cli["client_talk_type"], cli["mi_quality"], normalize="columns")],
    axis=1,
).round(3)
print(ct.to_string())

print("\nlabel/role consistency (should be 0):",
      (ther["client_talk_type"] != "n/a").sum(), (cli["main_therapist_behaviour"] != "n/a").sum())

interlocutor share: {'therapist': 0.503, 'client': 0.497}

Therapist behaviour (row %, overall and by mi_quality):
                            all   high    low
main_therapist_behaviour                     
other                     0.325  0.329  0.281
question                  0.284  0.280  0.320
reflection                0.265  0.285  0.070
therapist_input           0.126  0.106  0.329

Client talk type (overall and by mi_quality):
                    all   high    low
client_talk_type                     
neutral           0.644  0.641  0.678
change            0.244  0.251  0.172
sustain           0.112  0.109  0.150

label/role consistency (should be 0): 0 0


In [8]:
print("words per utterance by role and quality:")
print(df.groupby(["interlocutor", "mi_quality"])["words"].describe().round(1).to_string())

print("\nwords per utterance by therapist behaviour:")
print(ther.groupby("main_therapist_behaviour")["words"].describe().round(1).to_string())

print("\nwords per utterance by client talk type:")
print(cli.groupby("client_talk_type")["words"].describe().round(1).to_string())

print("\ntherapist share of spoken words, per conversation (median):")
share = df.pivot_table(index="transcript_id", columns="interlocutor", values="words", aggfunc="sum")
share["therapist_share"] = share["therapist"] / (share["therapist"] + share["client"])
print(share["therapist_share"].groupby(conv["mi_quality"]).median().round(3).to_string())

words per utterance by role and quality:
                          count  mean   std  min  25%   50%   75%    max
interlocutor mi_quality                                                 
client       high        4398.0  14.9  20.6  1.0  2.0   7.0  19.0  246.0
             low          419.0  15.4  18.3  1.0  2.0   8.0  21.5  111.0
therapist    high        4441.0  16.0  19.6  1.0  2.0  10.0  22.0  190.0
             low          441.0  23.5  24.8  1.0  6.0  14.0  36.0  128.0

words per utterance by therapist behaviour:
                           count  mean   std  min   25%   50%   75%    max
main_therapist_behaviour                                                  
other                     1586.0   4.0   8.3  1.0   1.0   1.0   2.0  117.0
question                  1386.0  19.0  16.5  1.0   8.0  14.0  25.0  128.0
reflection                1296.0  19.8  19.4  1.0   7.8  14.0  26.0  189.0
therapist_input            614.0  37.4  28.3  1.0  16.0  32.0  50.0  190.0

words per utterance by cl

## 4. Therapist -> client dynamics

Does the therapist behaviour immediately before a client turn change what kind of client talk follows? This is the kind of signal an adherence reviewer should surface.

In [9]:
df["prev_behaviour"] = df.groupby("transcript_id")["main_therapist_behaviour"].shift(1)
pairs = df[(df["interlocutor"] == "client") & (df["prev_behaviour"].notna()) & (df["prev_behaviour"] != "n/a")]

print("Client talk type given the preceding therapist behaviour (row %):")
for quality in [None, "high", "low"]:
    sub = pairs if quality is None else pairs[pairs["mi_quality"] == quality]
    print(f"\n-- {quality or 'all'} (n={len(sub)})")
    print(pd.crosstab(sub["prev_behaviour"], sub["client_talk_type"], normalize="index").round(3).to_string())

Client talk type given the preceding therapist behaviour (row %):

-- all (n=4785)
client_talk_type  change  neutral  sustain
prev_behaviour                            
other              0.233    0.650    0.116
question           0.290    0.593    0.117
reflection         0.233    0.665    0.103
therapist_input    0.196    0.692    0.112

-- high (n=4367)
client_talk_type  change  neutral  sustain
prev_behaviour                            
other              0.238    0.647    0.115
question           0.308    0.574    0.118
reflection         0.234    0.661    0.104
therapist_input    0.191    0.733    0.076

-- low (n=418)
client_talk_type  change  neutral  sustain
prev_behaviour                            
other              0.174    0.688    0.138
question           0.135    0.759    0.106
reflection         0.167    0.800    0.033
therapist_input    0.210    0.558    0.232


## 5. Per-conversation MI metrics (MITI-style summary scores)

Aggregate the utterance labels into a few conversation-level numbers (reflection-to-question ratio, % reflections, % change talk, ...) and check how well they separate high- from low-quality MI.

In [13]:
beh = pd.crosstab(ther["transcript_id"], ther["main_therapist_behaviour"])
tal = pd.crosstab(cli["transcript_id"], cli["client_talk_type"])

m = pd.DataFrame(index=conv.index)
m["mi_quality"] = conv["mi_quality"]
m["reflection_per_question"] = beh["reflection"] / beh["question"].where(beh["question"] > 0)
m["pct_reflection"] = beh["reflection"] / beh.sum(axis=1)
m["pct_question"] = beh["question"] / beh.sum(axis=1)
m["pct_input"] = beh["therapist_input"] / beh.sum(axis=1)
m["pct_change_talk"] = tal["change"] / tal.sum(axis=1)
m["pct_sustain_talk"] = tal["sustain"] / tal.sum(axis=1)
m["change_minus_sustain"] = m["pct_change_talk"] - m["pct_sustain_talk"]
m["therapist_word_share"] = share["therapist_share"]
metric_cols = [c for c in m.columns if c != "mi_quality"]

print(m.groupby("mi_quality")[metric_cols].median().round(3).T.to_string())


def auc(metric):
    """Probability a random high-quality conversation scores above a random low-quality one."""
    hi = m.loc[m["mi_quality"] == "high", metric].dropna()
    lo = m.loc[m["mi_quality"] == "low", metric].dropna()
    return ((hi.values[:, None] > lo.values[None, :]).mean()
            + 0.5 * (hi.values[:, None] == lo.values[None, :]).mean())


print("\nAUC (high > low) per metric:")
print(pd.Series({c: auc(c) for c in metric_cols}).round(3).sort_values(ascending=False).to_string())

mi_quality                high    low
reflection_per_question  1.000  0.000
pct_reflection           0.268  0.000
pct_question             0.279  0.250
pct_input                0.083  0.500
pct_change_talk          0.242  0.100
pct_sustain_talk         0.086  0.267
change_minus_sustain     0.129 -0.100
therapist_word_share     0.539  0.641

AUC (high > low) per metric:
pct_reflection             0.925
reflection_per_question    0.894
change_minus_sustain       0.742
pct_change_talk            0.697
pct_question               0.603
pct_sustain_talk           0.364
therapist_word_share       0.353
pct_input                  0.080


## 6. Sizing for LLM / Content Understanding design

In [14]:
conv["approx_tokens"] = (conv["words"] * 1.35).round()  # rough English words -> tokens
print(conv[["n_utts", "duration_min", "words", "approx_tokens"]].describe(percentiles=[.5, .9, .99]).round(1).to_string())
print("\nconversations over 32k approx tokens:", (conv["approx_tokens"] > 32_000).sum())
print("total audio/video minutes (by last timestamp):", round(conv["duration_min"].sum()))

print("\nTwo shortest/longest transcripts (id, minutes, utterances):")
print(conv.sort_values("duration_min").iloc[[0, 1, -2, -1]][["mi_quality", "title", "duration_min", "n_utts"]].to_string())

       n_utts  duration_min   words  approx_tokens
count   133.0         133.0   133.0          133.0
mean     72.9           7.7  1153.6         1557.4
std      84.3           7.1  1067.5         1441.1
min       6.0           0.8   102.0          138.0
50%      47.0           5.5   901.0         1216.0
90%     168.6          15.9  2217.8         2994.0
99%     389.8          35.4  5014.2         6768.8
max     598.0          37.8  7347.0         9918.0

conversations over 32k approx tokens: 0
total audio/video minutes (by last timestamp): 1028

Two shortest/longest transcripts (id, minutes, utterances):
              mi_quality                                                       title  duration_min  n_utts
transcript_id                                                                                             
69                   low                                       Provider Counseling-1      0.850000       7
11                  high  Motivational Interviewing in Juvenile Ju

In [15]:
def show(tid, start=0, n=14):
    sub = df[df["transcript_id"] == tid].iloc[start:start + n]
    for r in sub.itertuples():
        label = r.main_therapist_behaviour if r.interlocutor == "therapist" else r.client_talk_type
        print(f"[{r.timestamp}] {r.interlocutor[:3].upper()} <{label}> {r.utterance_text}")


for q in ["high", "low"]:
    tid = conv[conv["mi_quality"] == q].index[0]
    print(f"\n===== {q} quality | transcript {tid} | {conv.loc[tid, 'title']} =====")
    show(tid)


===== high quality | transcript 0 | NEW VIDEO: Brief intervention: "Barbara" =====
[00:00:13] THE <question> Thanks for filling it out. We give this form to everyone once a year regardless of why they come in. It helps us provide better care. Is it okay if I take a look at what you put down?
[00:00:24] CLI <neutral> Sure.
[00:00:25] THE <therapist_input> So, let's see. It looks that you put-- You drink alcohol at least four times a week on average-
[00:00:34] CLI <neutral> Mm-hmm.
[00:00:34] THE <therapist_input> -and you usually have three to four drinks when you do drink.
[00:00:39] CLI <neutral> Usually three drinks and glasses of wine.
[00:00:42] THE <therapist_input> Okay. That's at least 12 drinks a week.
[00:00:46] CLI <neutral> Something like that.
[00:00:47] THE <therapist_input> Okay. Just so you know, my role, um, when we talk about alcohol use, is just to share information about risk and to help patients who want help. This is different than telling them what I think they 

## Key takeaways for the design

- **Size**: 133 transcripts, 9,699 utterances, ~1,030 min. Median conversation is ~5.5 min / ~1.2k tokens, max ~10k tokens, so a whole session fits in one LLM call (no chunking needed for scoring).
- **Imbalance**: 110 high vs 23 low quality; do not report accuracy alone, use per-class precision/recall or AUC. Low-quality transcripts are also shorter (median 3.4 vs 6 min).
- **Labels**: therapist = reflection / question / therapist_input / other; client = change / neutral / sustain. `"n/a"` is a real value, load with `keep_default_na=False`.
- **Strongest quality signals**: % reflections (AUC ~0.93), reflection-to-question ratio (~0.89), % therapist_input (low quality has far more, AUC ~0.08 i.e. inverted), change-minus-sustain talk (~0.74), therapist word share (high ~54% vs low ~64%).
- **Dynamics**: in high-quality MI, questions are followed by the most change talk (~31%); in low-quality MI, giving input is followed by the most sustain talk (~23%).
- **Gaps vs real MITI/MISC**: no simple vs complex reflection, no empathy/partnership/evocation globals, no MI-adherent/non-adherent flags. Only a coarse high/low label, so this dataset can calibrate and sanity-check, not fully replace a rubric.
- **Timestamps**: one start time per utterance (no end time); 8 transcripts have non-monotonic timestamps.

# ANNOMIfull-EDQA

In [3]:
import pandas as pd

# 1. Load the dataset
df = pd.read_csv("AnnoMI-full.csv")
df.head()

,mi_quality,transcript_id,video_title,video_url,topic,utterance_id,interlocutor,timestamp,utterance_text,annotator_id,therapist_input_exists,therapist_input_subtype,reflection_exists,reflection_subtype,question_exists,question_subtype,main_therapist_behaviour,client_talk_type
0,high,0,"NEW VIDEO: Brief intervention: ""Barbara""",https://www.youtube.com/watch?v=PaSKcfTmFEk,reducing alcohol consumption,0,therapist,00:00:13,Thanks for filling it out. We give this form to everyone once a year regardless of why they come in. It helps us pro...,3,False,NaN,False,NaN,True,open,question,NaN
1,high,0,"NEW VIDEO: Brief intervention: ""Barbara""",https://www.youtube.com/watch?v=PaSKcfTmFEk,reducing alcohol consumption,1,client,00:00:24,Sure.,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,neutral
2,high,0,"NEW VIDEO: Brief intervention: ""Barbara""",https://www.youtube.com/watch?v=PaSKcfTmFEk,reducing alcohol consumption,2,therapist,00:00:25,"So, let's see. It looks that you put-- You drink alcohol at least four times a week on average-",3,True,information,False,NaN,False,NaN,therapist_input,NaN
3,high,0,"NEW VIDEO: Brief intervention: ""Barbara""",https://www.youtube.com/watch?v=PaSKcfTmFEk,reducing alcohol consumption,3,client,00:00:34,Mm-hmm.,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,neutral
4,high,0,"NEW VIDEO: Brief intervention: ""Barbara""",https://www.youtube.com/watch?v=PaSKcfTmFEk,reducing alcohol consumption,4,therapist,00:00:34,-and you usually have three to four drinks when you do drink.,3,True,information,False,NaN,False,NaN,therapist_input,NaN


In [4]:
# 2. Unique annotators and their values
num_annotators = df["annotator_id"].nunique()
unique_annotators = df["annotator_id"].unique()

print(f"Total unique annotators: {num_annotators}")
print("Annotator IDs:", unique_annotators)

Total unique annotators: 10
Annotator IDs: [3 4 6 9 1 7 0 2 5 8]


In [5]:
# 3. Check if any transcript has been annotated by multiple annotators
annotators_per_transcript = df.groupby("transcript_id")["annotator_id"].nunique()

multi_annotated_transcripts = annotators_per_transcript[
    annotators_per_transcript > 1
]

if not multi_annotated_transcripts.empty:
    print(
        f"Found {len(multi_annotated_transcripts)} transcript(s) annotated by more than one annotator:\n"
    )

    # Display which annotators worked on these transcripts
    breakdown = (
        df.groupby("transcript_id")["annotator_id"]
        .unique()
        .loc[multi_annotated_transcripts.index]
    )
    print(breakdown)
else:
    print(
        "No transcripts have been annotated by different annotators (each transcript has exactly one annotator)."
    )

Found 7 transcript(s) annotated by more than one annotator:

transcript_id
7      [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
27     [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
55     [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
56     [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
66     [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
109    [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
130    [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Name: annotator_id, dtype: object


In [8]:
# Filter dataset for transcript_id 27
df_27 = df[df["transcript_id"] == 27]

# 1. Unique values and counts for 'talk_type'
print("--- talk_type counts ---")
print(df_27["client_talk_type"].value_counts(dropna=False))

print("\n" + "=" * 40 + "\n")

# 2. Unique values and counts for 'main_behaviour'
print("--- main_behaviour counts ---")
print(df_27["main_therapist_behaviour"].value_counts(dropna=False))

--- talk_type counts ---
client_talk_type
NaN        120
neutral     54
sustain     50
change       6
Name: count, dtype: int64


--- main_behaviour counts ---
main_therapist_behaviour
NaN                110
therapist_input     56
other               45
reflection          14
question             5
Name: count, dtype: int64


In [13]:
# Direct and fastest approach
df[["mi_quality", "transcript_id"]].value_counts()

mi_quality  transcript_id
high        56               1750
            7                 660
            55                650
            121               598
            109               460
                             ... 
low         32                 12
high        79                 11
            125                 7
low         69                  7
            59                  6
Name: count, Length: 133, dtype: int64

In [14]:
# Count unique transcripts for each quality level
df.groupby("mi_quality")["transcript_id"].nunique()

mi_quality
high    110
low      23
Name: transcript_id, dtype: int64